1. Load

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Preprocessing") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.csv("../data/raw/*.csv", header=True, inferSchema=True)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 20:41:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

2. Drop useless columns

In [2]:
df = df.drop("nameOrig", "nameDest")

3. Feature engineering

In [3]:
from pyspark.sql.functions import col

df = df.withColumn("deltaOrig", col("oldbalanceOrg") - col("newbalanceOrig"))
df = df.withColumn("deltaDest", col("newbalanceDest") - col("oldbalanceDest"))

df = df.withColumn(
    "isBalanceErrorOrig",
    (col("oldbalanceOrg") - col("amount") != col("newbalanceOrig")).cast("int")
)

df = df.withColumn(
    "isBalanceErrorDest",
    (col("oldbalanceDest") + col("amount") != col("newbalanceDest")).cast("int")
)

4. Log transform

In [4]:
from pyspark.sql.functions import log1p

df = df.withColumn("amount_log", log1p(col("amount")))

5. Encoding (type)

In [5]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="type", outputCol="type_index")
df = indexer.fit(df).transform(df)

6. Drop original categorical

In [6]:
df = df.drop("type")

7. Final Check

In [7]:
df.printSchema()
df.show(5)

root
 |-- step: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)
 |-- deltaOrig: double (nullable = true)
 |-- deltaDest: double (nullable = true)
 |-- isBalanceErrorOrig: integer (nullable = true)
 |-- isBalanceErrorDest: integer (nullable = true)
 |-- amount_log: double (nullable = true)
 |-- type_index: double (nullable = false)

+----+--------+-------------+--------------+--------------+--------------+-------+--------------+------------------+---------+------------------+------------------+-----------------+----------+
|step|  amount|oldbalanceOrg|newbalanceOrig|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|         deltaOrig|deltaDest|isBalanceErrorOrig|isBalanceErrorDest|       amount_log|type

8. Save

In [8]:
df = df.coalesce(2)

df.write.mode("overwrite").parquet("../data/processed/transactions")

In [9]:
df.select("isFraud").groupBy("isFraud").count().show()

[Stage 7:>                                                          (0 + 2) / 2]

+-------+-------+
|isFraud|  count|
+-------+-------+
|      1|   8213|
|      0|6354407|
+-------+-------+



In [10]:
df_saved = spark.read.parquet("../data/processed/transactions")
df_saved.printSchema()
df_saved.count()

root
 |-- step: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)
 |-- deltaOrig: double (nullable = true)
 |-- deltaDest: double (nullable = true)
 |-- isBalanceErrorOrig: integer (nullable = true)
 |-- isBalanceErrorDest: integer (nullable = true)
 |-- amount_log: double (nullable = true)
 |-- type_index: double (nullable = true)



6362620

In [11]:
print("Final columns:", len(df.columns))

Final columns: 14
